# EDA — Judol Detection (SEMMA Sample + Explore)

Notebook eksplorasi data anotasi untuk skripsi *Pengembangan Sistem Deteksi
Promosi Judi Online Berbasis Deep Learning untuk Mengatasi Manipulasi Unicode
dengan Adversarial Training*.

**Mengikuti** `docs/research/eda-plan.md` (Langkah A–G).
**Engine analisis Unicode:** `silverspeak` + `unicodedata`.
**Target runtime:** Google Colab (mount Google Drive).

> **Catatan scope:** analisis kualitas anotasi (Cohen's Kappa, inter-rater)
> sengaja TIDAK ada di notebook ini — sesi khusus. EDA memakai `label_final`
> apa adanya. Cohen's Kappa tetap metode evaluasi anotasi metodologi skripsi.
> Notebook dapat dijalankan **bertahap per section** (lihat daftar isi).

## Daftar Isi
- 0. Setup (Drive, dependensi, konfigurasi)
- Langkah A — Load & Audit Data Dasar
- Langkah B — Karakteristik Teks Standar
- Langkah B2 — Analisis Label Leakage via Fitur Struktural (RISIKO)
- Langkah C — Analisis Keyword & Leksikon
- Langkah C2 — Deep-Dive Kualitatif per Kelas
- Langkah D — Karakter Non-ASCII: Cakupan & Distribusi (INTI)
- Langkah E — Inspeksi Manual Pola Manipulasi
- Langkah F — Analisis Dampak terhadap Tokenizer (IndoBERT)
- Langkah G — Uji Awal Model Sederhana (opsional)


## 0. Setup

### 0.1 Mount Google Drive

Dataset `dataset_final.xlsx` dan artefak EDA disimpan di Google Drive.
Ubah `DRIVE_DATA_DIR` dan `DRIVE_OUTPUT_DIR` sesuai struktur Drive Anda.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === KONFIGURASI PATH (ubah sesuai struktur Drive Anda) ===
DRIVE_DATA_DIR    = '/content/drive/MyDrive/judol_detection/data/annotation'
DRIVE_OUTPUT_DIR  = '/content/drive/MyDrive/judol_detection/experiments/reports'
DRIVE_FIGURE_DIR  = '/content/drive/MyDrive/judol_detection/experiments/reports/figures'

DATASET_PATH      = f'{DRIVE_DATA_DIR}/dataset_final_simple.xlsx'   # 2 kolom (text_raw, label)
DATASET_FULL_PATH = f'{DRIVE_DATA_DIR}/dataset_final.xlsx'          # 7 kolom lengkap

import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_FIGURE_DIR, exist_ok=True)
print('Output dir :', DRIVE_OUTPUT_DIR)
print('Figure dir :', DRIVE_FIGURE_DIR)


### 0.2 Install Dependensi

Jalankan sekali per sesi Colab.

In [ ]:
%pip install -q pandas openpyxl matplotlib seaborn wordcloud scikit-learn silverspeak
# Langkah F (tokenizer IndoBERT) — install terpisah karena besar
# %pip install -q transformers

### 0.3 Import & Konfigurasi Global

In [ ]:
import warnings, json, re, unicodedata, hashlib
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RNG_SEED = 42
np.random.seed(RNG_SEED)

OUTPUT = Path(DRIVE_OUTPUT_DIR)
FIG    = Path(DRIVE_FIGURE_DIR)

def save_fig(name):
    p = FIG / name
    plt.tight_layout()
    plt.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    print('saved:', p)

# Helper simpan ringkasan teks ke Drive
summary_lines = []
def add_summary(s):
    summary_lines.append(s)
    print(s)
def flush_summary(name='eda-summary.md'):
    (OUTPUT / name).write_text('\n'.join(summary_lines), encoding='utf-8')
    print('flushed:', OUTPUT / name)

print('Setup OK')


### 0.4 Verifikasi Engine `silverspeak`

Catat versi untuk reproducibility (wajib menurut `eda-plan.md` Langkah D).

In [ ]:
# API silverspeak v3.0.0 (terverifikasi 2026-07-24). NOTE: v3 mengubah API total
# dari versi lama yang didokumentasikan Context7 - normalize_text TIDAK ada lagi.
import silverspeak
print('silverspeak version:', getattr(silverspeak, '__version__', 'unknown (no __version__)'))
from silverspeak.homoglyphs import load_default_kb
from silverspeak.homoglyphs.fast_normalize import normalize_fast
from silverspeak.homoglyphs.utils import TypesOfHomoglyphs
print('TypesOfHomoglyphs:', [t.name for t in TypesOfHomoglyphs])

# Warm-up knowledge base sekali (graph homoglyph). Dipakai Langkah D & E.
KB = load_default_kb()
GRAPH_PATH = KB.graph_path
print('KB loaded, graph:', GRAPH_PATH)

# Sanity test end-to-end: teks dengan homoglyph Cyrillic-a & Greek-omicron
_test = 'sl\u03bft g\u0430cor b\u03bfnus'
_res = normalize_fast(_test, GRAPH_PATH, min_score=0.0, score_margin=0.0)
print('sanity test -> text:', repr(_res.text), '| changes:', len(_res.chars_changed))
assert _res.text == 'slot gacor bonus', 'sanity test gagal - cek versi silverspeak'
print('silverspeak OK')


---

## Langkah A — Load & Audit Data Dasar

**Tujuan:** konfirmasi integritas dataset.

In [ ]:
df = pd.read_excel(DATASET_PATH, sheet_name='Sheet1')
print('shape:', df.shape)
print('cols :', list(df.columns))
df.head(3)


In [ ]:
# Distribusi kelas
add_summary('# EDA Summary - Langkah A\n')
add_summary(f'Total rows: {len(df)}')
add_summary(f'Label distribution: {df["label"].value_counts().to_dict()}')
add_summary(f'Exact duplicate text_raw: {int(df["text_raw"].duplicated().sum())}')
add_summary(f'Null/empty text_raw: {int(df["text_raw"].isna().sum())} null, {int((df["text_raw"].fillna("").str.strip()=="").sum())} empty')
add_summary('NOTE: balanced 1029:1029 artificial -> baseline majority = 50%, akurasi mentah tidak informatif')

fig, ax = plt.subplots(figsize=(5,4))
df['label'].value_counts().plot(kind='bar', ax=ax, color=['#4a90d9','#d94a4a'])
ax.set_xticklabels(['0 (non-judol)','1 (judol)'], rotation=0)
ax.set_title('Distribusi Kelas (balanced)')
save_fig('01a-class-distribution.png')


---

## Langkah B — Karakteristik Teks Standar

**Tujuan:** memahami bentuk teks sebelum aspek Unicode.

In [ ]:
df['t']      = df['text_raw'].fillna('').astype(str)
df['n_char'] = df['t'].str.len()
df['n_word'] = df['t'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12,4))
for lbl, ax, ttl in [(0,axes[0],'non-judol'),(1,axes[1],'judol')]:
    sub = df[df.label==lbl]['n_char']
    ax.hist(sub, bins=40, color='#4a90d9' if lbl==0 else '#d94a4a', alpha=0.8)
    ax.set_title(f'Panjang teks (char) - {ttl}\nmedian={sub.median():.0f}, mean={sub.mean():.1f}')
    ax.set_xlabel('jumlah karakter')
save_fig('01b-length-by-class.png')

add_summary('\n## Langkah B - statistik teks')
for lbl in [0,1]:
    sub = df[df.label==lbl]['n_char']
    add_summary(f'kelas {lbl}: char median={sub.median():.0f}, mean={sub.mean():.1f}, std={sub.std():.1f}')


In [ ]:
# Elemen struktural (precompute untuk Langkah B2)
URL_RE     = re.compile(r'https?://|\.(?:com|org|net|id|co|info|link)\b|\blink\b', re.I)
MENTION_RE = re.compile(r'@\w{3,}')
PHONE_RE   = re.compile(r'\b08\d{8,}|\b62\d{8,}|wa\.me|whatsapp', re.I)
def struct_features(t):
    return pd.Series({
        'has_url'         : int(bool(URL_RE.search(t))),
        'has_mention'     : int(bool(MENTION_RE.search(t))),
        'has_phone_wa'    : int(bool(PHONE_RE.search(t))),
        'n_emoji'         : sum(1 for c in t if ord(c) > 0x1F000),
        'n_uppercase_word': sum(1 for w in t.split() if w.isupper() and len(w) > 2),
        'n_punct'         : sum(1 for c in t if c in '!?.,\;:\"\'()[]{}/'),
        'digit_ratio'     : (sum(c.isdigit() for c in t) / max(len(t),1)),
    })
sfeat = df['t'].apply(struct_features)
df_feat = pd.concat([df[['label']], sfeat], axis=1)
df_feat.groupby('label').mean().round(3)


---

## Langkah B2 - Analisis Label Leakage via Fitur Struktural (RISIKO KRITIS)

**Motivasi (inspeksi awal):** emoji judol ~51% vs non-judol ~13%; uppercase word
judol ~10 vs non-judol ~2. Bila fitur dangkal ini bisa memisahkan kelas, model
berisiko belajar shortcut, bukan robustness Unicode.

**Output:** AUC per fitur + F1 baseline shallow -> menentukan kontrol split.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

add_summary('\n## Langkah B2 - label leakage\n')
add_summary('| feature | AUC ROC |')
add_summary('|---|---|')
leak_feats = ['has_url','has_mention','has_phone_wa','n_emoji','n_uppercase_word','n_punct','digit_ratio']
X = df_feat[leak_feats].values
y = df_feat['label'].values
aucs = {}
for i, f in enumerate(leak_feats):
    try:
        a = roc_auc_score(y, X[:, i])
        aucs[f] = a
        flag = '  <-- RISK' if abs(a-0.5) > 0.35 else ''
        add_summary(f'| {f} | {a:.3f}{flag} |')
    except Exception as e:
        add_summary(f'| {f} | ERR {e} |')

# Baseline shallow: fitur struktural saja, NO teks
f1 = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5, scoring='f1').mean()
acc = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5, scoring='accuracy').mean()
add_summary(f'\nBaseline shallow (fitur struktural, no teks): F1={f1:.3f}, Acc={acc:.3f}')
add_summary('Interpretasi: ini UPPER BOUND shortcut. Bila tinggi, robustness Unicode mungkin tidak banyak menolong tanpa kontrol fitur.')

order = sorted(aucs, key=lambda k: abs(aucs[k]-0.5), reverse=True)
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(order, [aucs[f]-0.5 for f in order],
        color=['#d94a4a' if abs(aucs[f]-0.5)>0.35 else '#4a90d9' for f in order])
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('AUC - 0.5  (negatif = prediktor kelas 0)')
ax.set_title('Separability fitur struktural (|AUC-0.5| > 0.35 = risiko)')
save_fig('01b2-feature-leakage-auc.png')


In [ ]:
# Simpan laporan leakage terpisah
leak_md = ['# Label Leakage Analysis (Langkah B2)\n',
           f'Baseline shallow F1={f1:.3f}, Acc={acc:.3f}\n',
           '\n| feature | AUC | risk |\n|---|---|---|']
for f in order:
    risk = 'YES' if abs(aucs[f]-0.5) > 0.35 else 'no'
    leak_md.append(f'| {f} | {aucs[f]:.3f} | {risk} |')
leak_md.append('\nKeputusan kontrol split: tulis manual setelah melihat hasil di atas.')
(OUTPUT/'eda-feature-leakage.md').write_text('\n'.join(leak_md), encoding='utf-8')
print('saved:', OUTPUT/'eda-feature-leakage.md')


---

## Langkah C - Analisis Keyword & Leksikon

**Tujuan:** daftar kata target untuk attack generator (Tahap 1 adversarial plan).

In [ ]:
def tokenize(t):
    return re.findall(r'[a-z0-9]+', t.lower())

df['tokens'] = df['t'].apply(tokenize)

# Frekuensi kata per kelas
freq_pos = Counter(w for toks in df[df.label==1]['tokens'] for w in toks)
freq_neg = Counter(w for toks in df[df.label==0]['tokens'] for w in toks)
n_pos = len(df[df.label==1]); n_neg = len(df[df.label==0])

# Log-odds ratio dengan smoothing
def log_odds(w):
    a = freq_pos.get(w,0)+0.5; b = freq_neg.get(w,0)+0.5
    return np.log((a/n_pos)/(b/n_neg))

vocab = set(freq_pos)|set(freq_neg)
vocab = [w for w in vocab if len(w) > 2 and (freq_pos.get(w,0)+freq_neg.get(w,0)) >= 10]
lo = [(w, log_odds(w), freq_pos.get(w,0), freq_neg.get(w,0)) for w in vocab]
judol_kw = sorted(lo, key=lambda x: -x[1])[:30]
nonjudol_kw = sorted(lo, key=lambda x: x[1])[:30]

add_summary('\n## Langkah C - keyword\n')
add_summary('### Top 30 keyword khas JUDOL (log-odds tinggi):')
add_summary(', '.join(w for w,_,_,_ in judol_kw))
add_summary('\n### Top 30 keyword khas NON-JUDOL (log-odds rendah):')
add_summary(', '.join(w for w,_,_,_ in nonjudol_kw))

# Simpan keywords.json untuk Tahap 1
keywords_json = {
    'judol_target_words':  [w for w,_,_,_ in judol_kw],
    'nonjudol_trap_words': [w for w,_,_,_ in nonjudol_kw],
}
(OUTPUT/'eda-keywords.json').write_text(json.dumps(keywords_json, indent=2, ensure_ascii=False), encoding='utf-8')
print('saved:', OUTPUT/'eda-keywords.json')
print('Top judol:', [w for w,_,_,_ in judol_kw[:15]])


In [ ]:
from wordcloud import WordCloud
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for ax, (freq, ttl, cmap) in zip(axes, [(freq_pos,'judol','Reds'),(freq_neg,'non-judol','Blues')]):
    wc = WordCloud(width=600, height=400, background_color='white', colormap=cmap,
                   max_words=80).generate_from_frequencies(freq)
    ax.imshow(wc); ax.axis('off'); ax.set_title(ttl)
save_fig('01c-wordcloud-by-class.png')


---

## Langkah C2 - Deep-Dive Kualitatif per Kelas

**Manual work:** close reading 20-30 sampel per kelas. Output ditulis ke
`eda-qualitative-notes.md` secara manual berdasarkan tabel sampel di bawah.

In [ ]:
# Stratified random sample untuk close reading
n_per = 25
sample_pos = df[df.label==1].sample(n_per, random_state=RNG_SEED)[['t']]
sample_neg = df[df.label==0].sample(n_per, random_state=RNG_SEED)[['t']]
pd.set_option('display.max_colwidth', 200)
print('=== SAMPLE JUDOL (untuk close reading) ===')
for i, row in sample_pos.iterrows():
    print(f'[{i}]', row['t'][:180])
print('\n=== SAMPLE NON-JUDOL (untuk close reading) ===')
for i, row in sample_neg.iterrows():
    print(f'[{i}]', row['t'][:180])


In [ ]:
# Hard negatives: kelas 0 yang mengandung >=2 keyword judol baseline
BASELINE_KW = ['slot','gacor','wd','maxwin','togel','link','situs','daftar','judi','deposit','bonus','jackpot']
def n_baseline_kw(t):
    tl = t.lower()
    return sum(1 for k in BASELINE_KW if k in tl)
df['n_baseline_kw'] = df['t'].apply(n_baseline_kw)
hard_neg = df[(df.label==0) & (df.n_baseline_kw >= 2)].sort_values('n_baseline_kw', ascending=False)
print(f'Hard negatives (kelas 0, >=2 baseline keyword): {len(hard_neg)} dokumen')
print('Contoh (untuk audit annotation guidelines):')
for i, row in hard_neg.head(10).iterrows():
    print(f'  [{i}] ({row.n_baseline_kw} kw)', row['t'][:160])


In [ ]:
# Template catatan kualitatif - ISI MANUAL setelah close reading
qualitative_template = '''# Deep-Dive Kualitatif (Langkah C2)

## Pola naratif kelas JUDOL
- (isi: CTA, bukti WD, nama situs, instruksi deposit, dll)

## Pola naratif kelas NON-JUDOL
- (isi: alasan penolakan keyword sensitif, mis. "slot waktu", "bonus gaji")

## Hard negatives (kasus paling menantang untuk adversarial)
- (isi: index + alasan)

## Hard positives (eupemisme, inisial, emoji-only)
- (isi: index + pola)

## Target kata sensitif ber-konteks (input attack generator)
- mis. "slot" sensitif hanya dekat gacor/wd/link

## Usulan revisi annotation guidelines
- (isi)
'''
(OUTPUT/'eda-qualitative-notes.md').write_text(qualitative_template, encoding='utf-8')
print('Template disimpan ke:', OUTPUT/'eda-qualitative-notes.md')
print('-> ISI MANUAL setelah close reading sampel di atas')


---

## Langkah D - Karakter Non-ASCII: Cakupan & Distribusi (INTI)

Engine: `silverspeak` (homoglyph) + `unicodedata` (kategori karakter).
**Wajib laporkan DUA angka:** non-ASCII kasar vs non-ASCII mencurigakan.

In [ ]:
# Klasifikasi karakter non-ASCII ke kategori
def classify_char(ch):
    cp = ord(ch)
    if cp <= 0x7e: return 'ascii'
    try: name = unicodedata.name(ch, '')
    except Exception: name = ''
    cat = unicodedata.category(ch)
    # zero-width / format
    if cp in (0x200b,0x200c,0x200d,0xfeff,0x2060,0x180e) or cat=='Cf': return 'zero_width'
    # emoji / pictograph / symbol
    if cp >= 0x1F000 or cat.startswith('So') or 'EMOJI' in name or 'SYMBOL' in name: return 'emoji_symbol'
    if cat.startswith(('P','S')): return 'punct_symbol'  # tanda baca/simbol non-ASCII
    # whitespace non-ASCII
    if cat.startswith('Z'): return 'whitespace_nonascii'
    # huruf
    if 'LATIN' in name:
        if cp > 0x17f: return 'latin_extended'
        return 'latin_basic'
    # NON-LATIN lintas-skrip = kandidat homoglyph kuat
    if any(b in name for b in ('CYRILLIC','GREEK','ARMENIAN','CHEROKEE')):
        return 'cross_script_homoglyph_candidate'
    return 'other_nonlatin'

# Cakupan: % dokumen dengan >=1 char kategori X, per kelas
def char_categories(t):
    return set(classify_char(c) for c in t if ord(c) > 0x7e)

df['nonascii_cats'] = df['t'].apply(char_categories)
all_cats = ['emoji_symbol','cross_script_homoglyph_candidate','latin_extended',
            'punct_symbol','whitespace_nonascii','zero_width','other_nonlatin']

add_summary('\n## Langkah D - karakter non-ASCII\n')
add_summary('| kategori | % dok judol | % dok non-judol |')
add_summary('|---|---|---|')
cov = {}
for cat in all_cats:
    p1 = df[df.label==1]['nonascii_cats'].apply(lambda s: cat in s).mean()
    p0 = df[df.label==0]['nonascii_cats'].apply(lambda s: cat in s).mean()
    cov[cat] = (p1, p0)
    add_summary(f'| {cat} | {p1*100:.1f}% | {p0*100:.1f}% |')

# DUA angka wajib
any_nonascii_pos = df[df.label==1]['nonascii_cats'].apply(len).gt(0).mean()
any_nonascii_neg = df[df.label==0]['nonascii_cats'].apply(len).gt(0).mean()
susp_pos = df[df.label==1]['nonascii_cats'].apply(lambda s: bool(s & {'cross_script_homoglyph_candidate','zero_width'})).mean()
susp_neg = df[df.label==0]['nonascii_cats'].apply(lambda s: bool(s & {'cross_script_homoglyph_candidate','zero_width'})).mean()
add_summary('\n### DUA angka latar belakang (jangan satukan):')
add_summary(f'- Non-ASCII KASAR (apapun): judol={any_nonascii_pos*100:.1f}%, non-judol={any_nonascii_neg*100:.1f}%')
add_summary(f'- Non-ASCII MENCURIGAKAN (cross-script/zero-width): judol={susp_pos*100:.1f}%, non-judol={susp_neg*100:.1f}%')

fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(all_cats)); w = 0.4
ax.bar(x-w/2, [cov[c][0]*100 for c in all_cats], w, label='judol', color='#d94a4a')
ax.bar(x+w/2, [cov[c][1]*100 for c in all_cats], w, label='non-judol', color='#4a90d9')
ax.set_xticks(x); ax.set_xticklabels(all_cats, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('% dokumen'); ax.legend(); ax.set_title('Cakupan kategori karakter non-ASCII per kelas')
save_fig('01d-nonascii-coverage.png')


In [ ]:
# silverspeak v3.0.0: deteksi homoglyph via normalize_fast
# Output per dokumen: jumlah karakter yang berubah (chars_changed) + teks ternormalisasi.
def analyze_homoglyph(t):
    """Return (n_chars_changed, list_of_CharChange, normalized_text)."""
    try:
        res = normalize_fast(t, GRAPH_PATH, min_score=0.0, score_margin=0.0)
        return len(res.chars_changed), res.chars_changed, res.text
    except Exception:
        return 0, [], t

sp_out = df['t'].apply(analyze_homoglyph)
df['sp_n_changed']      = sp_out.str[0]
df['sp_chars_changed']  = sp_out.str[1]
df['sp_normalized']     = sp_out.str[2]
df['sp_has_homoglyph']  = df['sp_n_changed'] > 0

sp_pos = df[df.label==1]['sp_has_homoglyph'].mean()
sp_neg = df[df.label==0]['sp_has_homoglyph'].mean()
add_summary(f'\nsilverspeak v3 detect (normalize_fast, >=1 char berubah): judol={sp_pos*100:.1f}%, non-judol={sp_neg*100:.1f}%')

# Contoh konkret: tampilkan beberapa pasangan src->dst yang terdeteksi (untuk validasi Langkah E)
ex_changes = []
for _, row in df[df.sp_has_homoglyph].head(20).iterrows():
    for cc in row['sp_chars_changed'][:3]:
        ex_changes.append({'label': row['label'], 'src': cc.src, 'dst': cc.dst, 'score': round(cc.score,2)})
print('Contoh homoglyph terdeteksi (sample 20 dok, max 3/dok):')
print(pd.DataFrame(ex_changes).to_string(index=False) if ex_changes else '(tidak ada)')


---

## Langkah E - Inspeksi Manual Pola Manipulasi

**Manual work:** stratified sample 100-200 dokumen dengan cross-script/zero-width.
Klasifikasi manual: homoglyph sengaja / number-sub / noise / lainnya.

In [ ]:
# Sample dokumen yang mengandung karakter mencurigakan
susp_mask = df['nonascii_cats'].apply(lambda s: bool(s & {'cross_script_homoglyph_candidate','zero_width'}))
susp = df[susp_mask].copy()
print(f'Dokumen dengan karakter mencurigakan: {len(susp)} (judol: {(susp.label==1).sum()}, non-judol: {(susp.label==0).sum()})')

# Stratified sample (maks 100, seimbang)
n_samp = min(50, len(susp)//2) if len(susp) > 0 else 0
if n_samp > 0:
    samp = pd.concat([
        susp[susp.label==1].sample(min(n_samp,(susp.label==1).sum()), random_state=RNG_SEED),
        susp[susp.label==0].sample(min(n_samp,(susp.label==0).sum()), random_state=RNG_SEED),
    ])
    print(f'Sample untuk audit manual: {len(samp)}')

    # Ekstrak pasangan (karakter mencurigakan, konteks) untuk tiap dokumen
    audit_rows = []
    for idx, row in samp.iterrows():
        t = row['t']
        for pos, ch in enumerate(t):
            if classify_char(ch) in ('cross_script_homoglyph_candidate','zero_width'):
                ctx = t[max(0,pos-10):pos+11]
                try: name = unicodedata.name(ch)
                except Exception: name = '?'
                audit_rows.append({'sample_idx':idx,'label':row['label'],'char':ch,
                                   'char_name':name,'codepoint':f'U+{ord(ch):04X}','context':ctx})
    audit = pd.DataFrame(audit_rows)
    print('Total karakter mencurigakan di sample:', len(audit))
    audit.head(20)
else:
    audit = pd.DataFrame()
    print('Tidak ada dokumen mencurigakan - cek Langkah D')


In [ ]:
# Simpan template audit CSV - kategori DIISI MANUAL
if len(audit) > 0:
    audit['silverspeak_candidate'] = ''  # diisi via reverse mapping/manual
    audit['category'] = ''  # manual: intentional_homoglyph | number_sub | natural_noise | other
    audit['notes'] = ''
    audit.to_csv(OUTPUT/'eda-manual-audit.csv', index=False, encoding='utf-8')
    print('Template audit disimpan ke:', OUTPUT/'eda-manual-audit.csv')
    print('-> ISI kolom category MANUAL: intentional_homoglyph | number_sub | natural_noise | other')


---

## Langkah F - Analisis Dampak terhadap Tokenizer (IndoBERT)

**Bukti rumusan masalah:** mengapa model gagal (token rusak, bukan model bodoh).
IndoBERT dijalankan di Colab (besar, ~400MB). Uncomment install di 0.2 bila perlu.

In [ ]:
# %pip install -q transformers  # uncomment bila belum di 0.2
from transformers import AutoTokenizer
TOKENIZER_NAME = 'indobenchmark/indobert-base-p1'
tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
print('tokenizer:', TOKENIZER_NAME)

# Pasangan kata bersih vs homoglyph (omicron Greek U+03BF, a Cyrillic U+0430)
pairs = [
    ('slot',   'sl\u03bft'),     # o -> Greek omicron
    ('bonus',  'b\u03bfnus'),    # o -> Greek omicron
    ('gacor',  'g\u0430cor'),    # a -> Cyrillic a
    ('maxwin', 'm\u0430xwin'),   # a -> Cyrillic a
    ('link',   'l\u0131nk'),     # i -> dotless i
]
rows = []
for clean, obf in pairs:
    cid = tok(clean)['input_ids']
    oid = tok(obf)['input_ids']
    rows.append({
        'clean': clean, 'clean_tokens': tok.convert_ids_to_tokens(cid),
        'homoglyph': obf, 'homoglyph_tokens': tok.convert_ids_to_tokens(oid),
        'same_id': cid == oid,
        'has_UNK': tok.unk_token in tok.convert_ids_to_tokens(oid),
    })
import pandas as pd
pd.set_option('display.max_colwidth', 50)
tok_df = pd.DataFrame(rows)
tok_df


In [ ]:
add_summary('\n## Langkah F - kegagalan tokenizer IndoBERT\n')
add_summary('| kata bersih | token bersih | homoglyph | token homoglyph | UNK? | same ID? |')
add_summary('|---|---|---|---|---|---|')
for r in rows:
    add_summary(f"| {r['clean']} | {r['clean_tokens']} | {r['homoglyph']} | {r['homoglyph_tokens']} | {r['has_UNK']} | {r['same_id']} |")
flush_summary()
print('Ringkasan lengkap di:', OUTPUT/'eda-summary.md')


---

## Langkah G - Uji Awal Model Sederhana (opsional)

**Preliminary** bukti penurunan akurasi. Bisa ditunda ke Tahap 3/4 adversarial.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

X_tr, X_te, y_tr, y_te = train_test_split(df['t'], df['label'], test_size=0.2,
                                          stratify=df['label'], random_state=RNG_SEED)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
Xtr = vec.fit_transform(X_tr); Xte = vec.transform(X_te)
clf = LogisticRegression(max_iter=1000).fit(Xtr, y_tr)
f1_clean = f1_score(y_te, clf.predict(Xte))
print(f'Baseline TF-IDF + LogReg pada data uji bersih: F1={f1_clean:.3f}')

# Buat versi adversarial test: substitusi sederhana o->omicron pada keyword
def simple_obfuscate(t):
    for kw in ['slot','bonus','gacor','link','togel']:
        t = t.replace(kw, kw.replace('o','\u03bf').replace('a','\u0430'))
    return t
Xte_adv = vec.transform(X_te.apply(simple_obfuscate))
f1_adv = f1_score(y_te, clf.predict(Xte_adv))
print(f'Pada data uji adversarial (homoglyph sederhana): F1={f1_adv:.3f}')
print(f'Robustness gap awal: {f1_clean-f1_adv:.3f}  (preliminary, bukan hasil final)')


---

## Selesai - Checklist Artefak

Cek semua artefak tersimpan di Google Drive (sesuai `eda-plan.md` §6):

- [ ] `experiments/reports/eda-summary.md`
- [ ] `experiments/reports/eda-keywords.json`
- [ ] `experiments/reports/eda-feature-leakage.md`
- [ ] `experiments/reports/eda-qualitative-notes.md` (ISI MANUAL)
- [ ] `experiments/reports/eda-manual-audit.csv` (ISI MANUAL)
- [ ] `experiments/reports/figures/*.png`

**Langkah berikutnya** (di luar notebook ini):
1. Isi manual `eda-qualitative-notes.md` (Langkah C2) dan `eda-manual-audit.csv` (Langkah E).
2. Tulis `eda-split-decision.md` memakai hasil Langkah B2 (leakage) + dedup text-similarity.
3. Update `feature_list.json` (`EDA-001` -> `passing`) dan `claude-progress.md`.


In [ ]:
# Verifikasi semua artefak tersimpan
import os
expected = ['eda-summary.md','eda-keywords.json','eda-feature-leakage.md',
            'eda-qualitative-notes.md','eda-manual-audit.csv']
print('=== Artefak di', OUTPUT, '===')
for f in expected:
    p = OUTPUT/f
    print(f'  {"OK" if p.exists() else "MISSING"} {f}  ({p.stat().st_size if p.exists() else 0} bytes)')
print('=== Figures di', FIG, '===')
for f in sorted(os.listdir(FIG)):
    print(f'  OK {f}')
